In [1]:
import pymc as pm
import pandas as pd
import numpy as np
import arviz as az
import matplotlib.pyplot as plt
import pytensor as pt

df = pd.read_csv('full_data_bhm1.csv', keep_default_na=False, na_values=[''])
region_unemployment_rates = df.groupby('region')['stateUnemploymentRate'].mean()
df['regionUnemploymentRate'] = df['region'].map(region_unemployment_rates).round(1)
df

,name,yearEstablished,state,stateAbbreviation,locationType,locationTypeKind,typeOwned,typeProfit,level,hasUndergraduate,...,closedByRegion_2023,totalByState_2024,closedByState_2024,totalByRegion_2024,closedByRegion_2024,totalByState_2025,closedByState_2025,totalByRegion_2025,closedByRegion_2025,regionUnemploymentRate
0,Alderson Broaddus University,1871,West Virginia,WV,Rural,Distant,Private,Nonprofit,4-year,True,...,1,69,0,2055,4,69,0,2055,2,3.6
1,Alliance University (Formerly Nyack College),1882,New York,NY,City,Large,Private,Nonprofit,4-year,True,...,4,413,3,1163,8,413,0,1163,0,4.1
2,Ancilla College,1937,Indiana,IN,Rural,Distant,Private,Nonprofit,2-year,True,...,6,102,1,1323,9,102,0,1323,4,4.1
3,Becker College,1784,Massachusetts,MA,City,Midsize,Private,Nonprofit,4-year,True,...,4,144,1,1163,8,144,0,1163,0,4.1
4,Bloomfield College,1868,New Jersey,NJ,Suburb,Large,Private,Nonprofit,4-year,True,...,4,144,0,1163,8,144,0,1163,0,4.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60,Eastern Nazarene College,1900,Massachusetts,MA,Suburb,Large,Private,Nonprofit,4-year,True,...,4,144,1,1163,8,144,0,1163,0,4.1
61,Cox College,1907,Missouri,MO,City,Midsize,Private,Nonprofit,4-year,True,...,6,137,0,1323,9,137,2,1323,4,4.1
62,University of Wisconsin–Platteville Richland,1967,Wisconsin,WI,Town,Distant,Public,Nonprofit,4-year,True,...,6,88,4,1323,9,88,1,1323,4,4.1
63,Maryland University of Integrative Health,1974,Maryland,MD,Suburb,Large,Private,Nonprofit,graduate school,False,...,1,77,0,2055,4,77,1,2055,2,3.6


In [8]:
years = [i for i in range(2020, 2026)]
states = df['state'].unique()
regions = df['region'].unique()

analysis_df = pd.read_csv('analysis_results.csv')

lda_results = analysis_df[[
    'name', 'lda_reason_component_1', 'lda_reason_component_2',
    'lda_reason_component_3', 'lda_reason_component_4', 
    'lda_reason_component_5', 'lda_reason_component_6'
]].copy()

cluster_results = analysis_df[['name', 'kmeans_cluster']].copy()

lda_results.rename(columns={
    'lda_reason_component_1': 'lda_class_0',
    'lda_reason_component_2': 'lda_class_1',
    'lda_reason_component_3': 'lda_class_2',
    'lda_reason_component_4': 'lda_class_3',
    'lda_reason_component_5': 'lda_class_4',
    'lda_reason_component_6': 'lda_class_5'
}, inplace=True)

cluster_results.rename(columns={'kmeans_cluster': 'cluster_label'}, inplace=True)

lda_results['state'] = df['state']
lda_results['region'] = df['region']
cluster_results['state'] = df['state']
cluster_results['region'] = df['region']

lda_results.drop(columns=['name'], inplace=True)
cluster_results.drop(columns=['name'], inplace=True)

lda_results

,lda_class_0,lda_class_1,lda_class_2,lda_class_3,lda_class_4,lda_class_5,state,region
0,-6.438618,2.927537,-0.682517,3.725940,1.770490,0.447840,West Virginia,South
1,-7.475466,4.332237,-1.197146,0.191720,3.488590,-2.293371,New York,Northeast
2,-22.178046,11.943267,-8.125005,-0.843801,-2.551320,-0.622213,Indiana,Midwest
3,-28.274857,-0.775211,20.803050,-0.405365,-2.892905,0.282596,Massachusetts,Northeast
4,-5.817318,5.151647,-0.365333,3.332977,3.710251,0.143425,New Jersey,Northeast
...,...,...,...,...,...,...,...,...
60,-7.604265,5.688804,1.963197,2.016482,2.376436,-0.331152,Massachusetts,Northeast
61,-5.380943,3.791087,1.757910,1.627719,0.118196,-0.416831,Missouri,Midwest
62,24.993869,-27.923197,-2.650798,3.227140,-2.197175,-0.450784,Wisconsin,Midwest
63,-21.358249,10.633362,-7.891616,-1.318980,-2.749178,-0.461946,Maryland,South


In [21]:
state_data = []
for year in years:
    for state in states:
        state_df_temp = df[df['state'] == state]
        total = state_df_temp[f'totalByState_{year}'].sum()
        closed = state_df_temp[f'closedByState_{year}'].sum()
        if total > 0:
            # Impute missing endowment with median
            # endowment_median = state_df_temp['endowmentMedian'].replace(-1, state_df_temp['endowmentMedian'].median())
            lda_row = lda_results[lda_results['state'] == state].iloc[0]
            lda_score = max(lda_row[['lda_class_0', 'lda_class_1', 'lda_class_2']].values)  # Adjust class names
            cluster_label = cluster_results[cluster_results['state'] == state]['cluster_label'].iloc[0]
            state_data.append({
                'state': state,
                'year': year,
                'total': total,
                'closed': closed,
                # 'endowmentMedian': endowment_median.mean(),
                'stateBirthRate': state_df_temp['stateBirthRate'].mean(),
                'stateUnemploymentRate': state_df_temp['stateUnemploymentRate'].mean(),
                'hasReligionAffiliation': state_df_temp['hasReligionAffiliation'].any(),
                'locationType': state_df_temp['locationType'].mode()[0],
                'lda_score': lda_score,
                'cluster_label': cluster_label
            })

state_df = pd.DataFrame(state_data)

location_type_map = {loc: i for i, loc in enumerate(state_df['locationType'].unique())}
location_type_idx = [location_type_map[loc] for loc in state_df['locationType']]
cluster_map = {cl: i for i, cl in enumerate(sorted(state_df['cluster_label'].unique()))}
cluster_idx = [cluster_map[cl] for cl in state_df['cluster_label']]

state_df

,state,year,total,closed,stateBirthRate,stateUnemploymentRate,hasReligionAffiliation,locationType,lda_score,cluster_label
0,West Virginia,2020,138,0,9.75,3.8,True,City,2.927537,20
1,New York,2020,2891,0,10.89,4.2,True,City,4.332237,19
2,Indiana,2020,204,0,11.79,3.9,True,City,11.943267,15
3,Massachusetts,2020,432,3,9.83,4.6,True,Suburb,20.803050,10
4,New Jersey,2020,144,0,11.09,4.8,True,Suburb,5.151647,6
...,...,...,...,...,...,...,...,...,...,...
163,Virginia,2025,138,0,11.15,3.3,True,Suburb,22.887706,16
164,Delaware,2025,16,0,10.58,3.9,True,City,1.550991,7
165,Oklahoma,2025,93,0,12.16,3.2,False,City,23.494255,7
166,Maryland,2025,77,1,11.25,3.1,False,Suburb,10.633362,16


In [22]:
with pt.config.change_flags(exception_verbosity='high'):
    with pm.Model() as model:
        # Global and hierarchical priors
        mu_global = pm.Normal('mu_global', mu=0, sigma=10)
        sigma_region = pm.HalfNormal('sigma_region', sigma=5)
        region_effects = pm.Normal('region_effects', mu=0, sigma=sigma_region, shape=len(regions))
        
        region_idx_map = {region: i for i, region in enumerate(regions)}
        state_to_region = dict(zip(df['state'], df['region']))
        state_region_idx = [region_idx_map[state_to_region[state_df.loc[i, 'state']]] for i in range(len(state_df))]
        
        sigma_state = pm.HalfNormal('sigma_state', sigma=5)
        state_effects = pm.Normal('state_effects', mu=region_effects[state_region_idx], sigma=sigma_state, shape=len(state_df))
        
        # Cluster effects
        sigma_cluster = pm.HalfNormal('sigma_cluster', sigma=2)
        cluster_effects = pm.Normal('cluster_effects', mu=0, sigma=sigma_cluster, shape=len(cluster_map))
        
        # Covariate effects
        # beta_endowment = pm.Normal('beta_endowment', mu=0, sigma=5)
        beta_birth = pm.Normal('beta_birth', mu=0, sigma=5)
        beta_unemp = pm.Normal('beta_unemp', mu=0, sigma=5)
        beta_religion = pm.Normal('beta_religion', mu=0, sigma=5)
        sigma_lda = pm.HalfNormal('sigma_lda', sigma=2)
        sigma_loc = pm.HalfNormal('sigma_loc', sigma=2)
        location_effects = pm.Normal('location_effects', mu=0, sigma=sigma_loc, shape=len(location_type_map))
        
        logit_p = (mu_global + 
                  region_effects[state_region_idx] + 
                  state_effects + 
                  cluster_effects[cluster_idx] + 
                #   beta_endowment * pm.math.log1pexp(state_df['endowmentMedian']) + 
                  beta_birth * pm.math.log1pexp(state_df['stateBirthRate'].values) + 
                  beta_unemp * pm.math.log1pexp(state_df['stateUnemploymentRate'].values) + 
                  beta_religion * state_df['hasReligionAffiliation'].astype(int).values + 
                  location_effects[location_type_idx] + 
                  state_df['lda_score'].values * sigma_lda)
        
        p = pm.Deterministic('p', pm.math.sigmoid(logit_p))
        y = pm.Binomial('y', n=state_df['total'], p=p, observed=state_df['closed'])
        
        trace = pm.sample(8000, tune=2000, target_accept=0.9, return_inferencedata=True, random_seed=42, progressbar=True)

az.to_netcdf(trace, 'good_bhm_traces/analysis_8k_2k_4.nc')

Initializing NUTS using jitter+adapt_diag...


SamplingError: Initial evaluation of model at starting point failed!
Starting values:
{'mu_global': array(0.3334881), 'sigma_region_log__': array(1.57349985), 'region_effects': array([ 0.76213487,  0.07250282,  0.27230549, -0.57786645]), 'sigma_state_log__': array(1.40323168), 'state_effects': array([ 1.5257522 ,  0.053349  ,  0.02318513, -0.78888442, -0.92470377,
        0.47982283,  0.43580508,  0.29339316,  1.16040398,  0.91213313,
        0.53216656, -0.88667392, -0.86230079, -0.10464079,  1.46917187,
        0.1614213 , -0.08352726, -0.57447116,  1.02286874, -1.36211493,
        0.78873724,  0.27215468, -1.23650516,  1.18207723,  0.31995725,
        0.02807704,  0.77959075,  0.77091784,  0.95006955, -0.2799889 ,
       -0.01446003,  0.62060058,  0.38335332, -0.08150833,  0.9404    ,
       -0.46266716,  0.78998885,  0.13669408,  1.34420948, -0.03838549,
       -1.39679665,  0.83306611,  1.4529506 ,  0.18314141, -0.65238528,
        0.7072922 ,  0.15428528,  0.15801003,  1.11195763

In [ ]:
# Summary and analysis
trace = az.from_netcdf('bhm/plan3_bhm_traces.nc')
print("=== Summary Statistics ===")
print(az.summary(trace, var_names=['mu_global', 'p', 'beta_endowment', 'beta_birth', 'beta_unemp', 'beta_religion', 'sigma_lda', 'cluster_effects'], hdi_prob=0.95))

az.plot_posterior(trace, var_names=['beta_endowment', 'beta_birth', 'beta_unemp', 'beta_religion', 'sigma_lda'], hdi_prob=0.95)
plt.tight_layout()
plt.show()